In [ ]:
if True:
  import importlib
  import DatasetBuilder
  import visualisation_mod
  import EDA
  import pandas as pd
  import numpy as np
  import torch
  import torch.nn as nn

  # Force a fresh load of both modules
  importlib.reload(DatasetBuilder)
  importlib.reload(visualisation_mod)
  importlib.reload(EDA)

  from EDA import ExploratoryAnalyzer
  from DatasetBuilder import MultimodalDatasetFactory, MultimodalUniverseFactory
  from visualisation_mod import UniverseVisualizer

  import matplotlib.pyplot as plt
  import seaborn as sns

  # ── Phase 2: Multimodal Fusion pipeline ─────────────────────────────
  import utils
  import multimodal_fusion
  importlib.reload(utils)
  importlib.reload(multimodal_fusion)
  from utils import (
      prepare_multimodal_data, compute_metrics,
      TABULAR_FEATURES, SpectralPCAExtractor,
  )
  from multimodal_fusion import (
      TabularMLP, SpectralMLP, MultimodalFusionModel,
      TrainConfig, train_model, build_dataloaders, evaluate,
  )

  from sklearn.model_selection import train_test_split
  from sklearn.preprocessing import StandardScaler

  DATA_ROOT = "/content/drive/MyDrive/Data_science_and_applications"
  factory = MultimodalUniverseFactory(base_path=DATA_ROOT)
  print("✓ Modules synced — fusion pipeline ready.")

# 🌌 Part 1 — Problem Definition: Absolute Magnitude via Late Fusion

###  Multi-Modal Extension Strategy

In Phase 1 (`deliverable1.ipynb`), we demonstrated that a neural network can successfully predict **Absolute Magnitude ($M$)** using proxies for distance derived strictly from tabular Gaia variables. The standard astronomical formula is:

$$M = m - 5 \log_{10}(d) + 5$$

Where $m$ is apparent magnitude and $d$ is distance. Because **distance ($d$)** is incredibly difficult to measure accurately across the universe, we now substitute non-tabular data — specifically Gaia BP/RP spectra (stored as 110 Chebyshev coefficients) — to capture intrinsic stellar luminosity indicators (gravity-sensitive absorption lines, chemical composition) that serve as **distance-independent proxies** for absolute magnitude.

### 🔬 The Late-Fusion Hypothesis
We implement a **multi-view "Late Fusion"** architecture:
- **Tabular Branch:** A Multi-Layer Perceptron (MLP) processing optimized distance-proxy tabular features (apparent magnitude, color index, temperature, metallicity, surface gravity).
- **Spectral Branch:** A linear / small MLP processing PCA-reduced spectral coefficients — the spectral envelope encodes the stellar class, which determines intrinsic luminosity.
- **Fusion Layer:** Concatenates both branch embeddings and feeds them into a final **Regression Head** that outputs the continuous variable: **Absolute Magnitude ($M$)**.

### 📊 Comparative Benchmark
We will train **three variations** on identical data splits:
1. **Tabular-only Model** — Phase-1 baseline (no spectral information).
2. **Spectral-only Model** — Can the spectral envelope alone predict $M$?
3. **Multimodal Fusion Model** — Does combining views beat both unimodal baselines?

Metrics: **RMSE ↓**, **R² ↑**, and **MAE ↓**.

## 💾 Part 2 — Raw Data Acquisition & Spectral Exploration

🔓 **Bypassing Pipeline Filters.** Unlike the restricted calibration slice used in Phase 1, our multimodal objective requires maximizing the statistical distribution. We stream the full Gaia dataset from HuggingFace's `MultimodalUniverse` without applying the astronomer-quality filters, then apply targeted cleaning to retain every star that has valid spectral coefficients (`coeff` column).

### The Spectral Modality: Chebyshev Coefficients
The Gaia BP/RP spectra are compressed into **110 Chebyshev polynomial coefficients** per star. These encode the full spectral energy distribution — from temperature-sensitive broad continuum slopes to narrow gravity- and metallicity-sensitive absorption features. Our goal is to distill these 110 dimensions into a compact **PCA latent space** (5 components) that captures the dominant variance of the spectral envelope.

In [ ]:
# ── 2a. Load raw data for exploration (factory-based, no filters) ──
factory = MultimodalUniverseFactory(base_path="MultimodalUniverse/gaia")
df_raw = factory.load_validated_dataset(
    path="MultimodalUniverse/gaia",
    n_wanted=40000,
    batch_size=200,
    filter_names=[]
)
print(f"✅ Raw acquisition complete. Shape: {df_raw.shape}")
print(f"   Total columns: {len(df_raw.columns)}")

# ── 2b. Run the full multimodal preparation pipeline ────────────────
# This streams data, computes abs_mag, imputes missing tabular features,
# extracts PCA components from the 110-D spectral coeffs, and splits.
print("\n⏳ Running multimodal data preparation pipeline …")
data = prepare_multimodal_data(
    path="MultimodalUniverse/gaia",
    n_wanted=50000,
    batch_size=500,
    pca_components=5,
    test_size=0.15,
    val_size=0.15,
    random_state=42,
)

print(f"\n✅ Multimodal pipeline complete:")
print(f"   Train: {data['n_train']}  |  Val: {data['n_val']}  |  Test: {data['n_test']}")
print(f"   Tabular features ({data['X_train_tab'].shape[1]}): {data['tabular_feature_names']}")
print(f"   PCA {data['pca_components']} components explain "
      f"{data['pca_extractor'].total_explained_variance_ * 100:.2f}% of spectral variance")
for i, ratio in enumerate(data['pca_extractor'].explained_variance_ratio_):
    print(f"     PC{i+1}: {ratio*100:.2f}%")

In [ ]:
# Inspect missing data across key features in the raw dataset
missing_summary = df_raw[['phot_g_mean_mag', 'bp_rp', 'teff_gspphot', 'parallax']].isnull().sum()
print("❌ Missing Values Count Per Key Feature:\n", missing_summary)

# The 'coeff' column contains the 110-D spectral coefficients — our non-tabular modality
coeff_available = df_raw['coeff'].notna().sum()
print(f"\n🔬 Stars with spectral coefficients (coeff): {coeff_available} / {len(df_raw)} "
      f"({coeff_available/len(df_raw)*100:.1f}%)")

# Target: Absolute Magnitude (computed internally by prepare_multimodal_data)
abs_mag_available = df_raw['abs_mag'].notna().sum()
print(f"✨ Stars with computable absolute magnitude: {abs_mag_available} / {len(df_raw)} "
      f"({abs_mag_available/len(df_raw)*100:.1f}%)")

## 🌌 Visualizing Spectral Coefficient Variations Across the HR Diagram

Because the 110 sequential coefficients represent mathematically compressed Chebyshev expansion shapes of the BP/RP spectrum, we extract three distinct stellar examples sorted by **Absolute Magnitude** ($M$) — a Luminous Giant, a Main-Sequence star, and a Faint Dwarf — to see how the spectral envelope changes with intrinsic brightness. This directly visualizes *why* the spectral modality carries distance-independent information about $M$.

In [ ]:
# ── Prepare data for spectral coefficient visualization ────────────
# Use the target (abs_mag) instead of distance to sort stars
selected_features = [
    'phot_g_mean_mag',
    'bp_rp',
    'pseudocolour',
    'rv_template_fe_h',
    'rv_template_logg'
]

# Build a clean subset with valid spectral + target data
df_viz = df_raw.dropna(subset=['abs_mag', 'coeff']).copy()
df_viz = df_viz[np.isfinite(df_viz['abs_mag'])]

# Impute missing tabular features for visualization context
for col in selected_features:
    if col in df_viz.columns and df_viz[col].isnull().sum() > 0:
        df_viz[col] = df_viz[col].fillna(df_viz[col].median())

X_viz_tab = df_viz[selected_features].values
X_viz_seq = np.stack(df_viz['coeff'].values)
y_viz_absmag = df_viz['abs_mag'].values

print("📐 Spectral Visualization Data Aligned:")
print(f" 🔹 Tabular Matrix shape   : {X_viz_tab.shape}  → (Stars, Features)")
print(f" 🔹 Spectral Coeffs shape  : {X_viz_seq.shape} → (Stars, 110 Coeffs)")
print(f" 🔹 Target (abs_mag) shape : {y_viz_absmag.shape}   → (Stars,)")
print(f" 🔹 abs_mag range          : [{y_viz_absmag.min():.1f}, {y_viz_absmag.max():.1f}]")

### 🔬 Spectral Signature Contrast by Luminosity Class

We sort stars by **Absolute Magnitude** and sample three regimes:
- **Luminous (bright absolute M):** Giants / supergiants — low surface gravity, deep absorption lines
- **Intermediate:** Main-sequence / turnoff stars
- **Faint (dim absolute M):** Dwarfs — high surface gravity, broadened lines

The coefficient profiles reveal how the spectral envelope systematically changes with intrinsic brightness — this is the physical signal our **Spectral Branch** learns to exploit.

In [ ]:
if True:
  # Sort by Absolute Magnitude (bright → faint)
  sorted_indices = np.argsort(y_viz_absmag)  # ascending: most negative (brightest) first

  idx_bright = sorted_indices[int(len(sorted_indices) * 0.05)]   # 5th percentile — Luminous
  idx_mid    = sorted_indices[int(len(sorted_indices) * 0.50)]   # 50th percentile — Intermediate
  idx_faint  = sorted_indices[int(len(sorted_indices) * 0.95)]   # 95th percentile — Faint

  seq_bright, mag_bright = X_viz_seq[idx_bright], y_viz_absmag[idx_bright]
  seq_mid,    mag_mid    = X_viz_seq[idx_mid],    y_viz_absmag[idx_mid]
  seq_faint,  mag_faint  = X_viz_seq[idx_faint],  y_viz_absmag[idx_faint]

  # Plot the discrete structural signatures
  plt.figure(figsize=(15, 6))

  plt.plot(seq_bright, label=f'Luminous Star  (M = {mag_bright:.1f})', color='#FFD700', alpha=0.9, linewidth=2)
  plt.plot(seq_mid,    label=f'Main-Sequence  (M = {mag_mid:.1f})',    color='teal', alpha=0.8, linewidth=2)
  plt.plot(seq_faint,  label=f'Faint Dwarf    (M = {mag_faint:.1f})',  color='purple', alpha=0.8, linewidth=2)

  plt.title('Gaia BP/RP Spectral Signature: 110-Coefficient Profile Contrast\n(Sorted by Absolute Magnitude — Intrinsic Brightness)', fontsize=13, fontweight='bold')
  plt.xlabel('Coefficient Sequence Position Index', fontsize=11)
  plt.ylabel('Chebyshev Coefficient Amplitude', fontsize=11)
  plt.yscale('symlog')  # Coefficients decay exponentially toward zero
  plt.grid(True, which="both", linestyle='--', alpha=0.5)
  plt.legend(fontsize=10, frameon=True)
  plt.tight_layout()
  plt.show()

## 📊 Feature-Target Correlation Analysis

Now that our objective is predicting **Absolute Magnitude ($M$)** rather than distance, we examine which tabular features show the strongest linear relationship with $M$. This informs which features carry the most predictive signal for our **Tabular Branch**.

In [ ]:
# Isolate numeric columns and compute Pearson correlation with Absolute Magnitude
numeric_df = df_raw.select_dtypes(include=[np.number])

absmag_correlations = numeric_df.corr()['abs_mag'].sort_values(ascending=False)

print("📊 TOP 10 POSITIVELY CORRELATED VARIABLES WITH ABSOLUTE MAGNITUDE (M):")
print(absmag_correlations.head(11))  # Includes the target itself

print("\n📊 TOP 10 NEGATIVELY CORRELATED VARIABLES WITH ABSOLUTE MAGNITUDE (M):")
print(absmag_correlations.tail(10))

### 🔭 The Physical Predictors (Distance-Independent Proxies)

These features form the input to our **Tabular Branch**. Critically, none of them is a direct distance measurement — they are all *observables* that correlate with intrinsic luminosity:

- **`phot_g_mean_mag`** ($m$, apparent magnitude): The observed brightness. When combined with a luminosity indicator, it constrains distance — but our model must learn this implicitly from the spectral branch.
- **`bp_rp`** (color index): Primary stellar temperature proxy. Hot stars (blue, low bp_rp) and cool stars (red, high bp_rp) have distinct absolute magnitudes at a given evolutionary stage.
- **`pseudocolour`** ($r \approx 0.46$ with M): Astrometric chromaticity — a critical parameter when standard color indices are missing. Encodes photon wavelength distribution.
- **`teff_gspphot`**: Effective temperature [K] from Gaia's GSP-Phot pipeline. Directly tied to the star's position on the HR diagram.
- **`rv_template_fe_h`** ([Fe/H], metallicity): Metal-poor stars (Population II) dominate the halo at large distances, creating a galactic-population gradient that correlates with intrinsic brightness.
- **`rv_template_logg`** ($\log g$, surface gravity): Giants ($\log g \approx 1$–$2$) vs. dwarfs ($\log g \approx 4$–$5$) differ by orders of magnitude in intrinsic luminosity — this is the strongest single-parameter luminosity indicator.

## 🧠 Part 3 — Model Architecture: Late-Fusion Design

We define three model variants, all built from the classes in `multimodal_fusion.py`. The data splits are already prepared in the `data` dictionary by `prepare_multimodal_data()`.

### Architecture Summary

| Variant | Input | Hidden Layers | Output |
|---|---|---|---|
| **Tabular-only** | 6 tabular features | 64→32→16 | $M$ |
| **Spectral-only** | 5 PCA components | 32→16→8 | $M$ |
| **Fusion** | 6 tab + 5 spec | TabEnc(32→16) + SpecEnc(16→16) → Concat(32) → 16→8→1 | $M$ |

The **Fusion model** encodes each modality into a 16-D embedding, concatenates them, and passes the joint representation through a shared regression head. This "late fusion" design lets each branch specialise before the final mixing layer.

In [ ]:
# ── Build all three model variants ──────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

tabular_dim = data['X_train_tab'].shape[1]
spectral_dim = data['X_train_spec'].shape[1]

model_tabular = TabularMLP(input_dim=tabular_dim)
model_spectral = SpectralMLP(input_dim=spectral_dim)
model_fusion = MultimodalFusionModel(
    tabular_dim=tabular_dim,
    spectral_dim=spectral_dim,
)

# Log parameter counts
for name, model in [("Tabular-only", model_tabular),
                     ("Spectral-only", model_spectral),
                     ("Fusion", model_fusion)]:
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {name:<18s}: {n_params:>6,d} parameters")

# ── Build DataLoaders for each modality mode ────────────────────────
config = TrainConfig()
config.device = str(device)

loaders_tab = build_dataloaders(data, config.batch_size, mode="tabular")
loaders_spec = build_dataloaders(data, config.batch_size, mode="spectral")
loaders_fusion = build_dataloaders(data, config.batch_size, mode="fusion")

print("✓ Models and DataLoaders ready for all 3 variants.")

## 🚀 Part 4 — Training: Three-Way Comparative Benchmark

We train all three variants under **identical conditions** (same data splits, same optimizer, same LR schedule). The `train_model()` function from `multimodal_fusion.py` handles the full training loop with:
- **Adam** optimizer ($\text{lr}=0.01$)
- **ReduceLROnPlateau** scheduler (factor 0.5, patience 8)
- **Gradient clipping** (max norm 1.0)
- **Early stopping** (24 epochs without improvement)
- **Best-weight restoration** before final testing

In [ ]:
# ── 4a. Train Tabular-only (Phase-1 baseline) ───────────────────────
print("▶ Variant 1/3: TABULAR-ONLY BASELINE")
results_tab = train_model(
    model_tabular, *loaders_tab, config, mode="tabular", model_name="Tabular-only",
)

# ── 4b. Train Spectral-only ──────────────────────────────────────────
print("\n▶ Variant 2/3: SPECTRAL-ONLY")
results_spec = train_model(
    model_spectral, *loaders_spec, config, mode="spectral", model_name="Spectral-only",
)

# ── 4c. Train Multimodal Fusion ──────────────────────────────────────
print("\n▶ Variant 3/3: MULTIMODAL FUSION")
results_fusion = train_model(
    model_fusion, *loaders_fusion, config, mode="fusion", model_name="Multimodal Fusion",
)

print("\n✓ All three variants trained.")

## 📊 Part 5 — Comparative Evaluation & Diagnostics

In [ ]:
# ── Comparative results table ───────────────────────────────────────
all_results = [results_tab, results_spec, results_fusion]

print(f"\n{'='*60}")
print(f"  COMPARATIVE BENCHMARK RESULTS")
print(f"{'='*60}")
print(f"{'Model':<22} {'RMSE ↓':>10} {'R²  ↑':>10} {'MAE ↓':>10}")
print(f"{'-'*54}")

best_rmse = float('inf')
best_model = ''
for r in all_results:
    m = r['test_metrics']
    print(f"{r['model_name']:<22} {m['rmse']:>10.4f} {m['r2']:>10.4f} {m['mae']:>10.4f}")
    if m['rmse'] < best_rmse:
        best_rmse = m['rmse']
        best_model = r['model_name']

print(f"{'-'*54}")

# Improvement over tabular baseline
base = results_tab['test_metrics']
for r in all_results:
    if r['model_name'] == 'Tabular-only':
        continue
    m = r['test_metrics']
    delta_rmse = m['rmse'] - base['rmse']
    delta_r2 = m['r2'] - base['r2']
    delta_mae = m['mae'] - base['mae']
    print(f"\n{r['model_name']} vs Tabular baseline:")
    print(f"  ΔRMSE = {delta_rmse:+.4f}  |  ΔR² = {delta_r2:+.4f}  |  ΔMAE = {delta_mae:+.4f}")

print(f"\n🏆 Best model: {best_model} (RMSE = {best_rmse:.4f})")
print(f"{'='*60}")

In [ ]:
# ── Diagnostic plots: Learning curves + Residual distributions ──────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

colors = {'Tabular-only': '#1f77b4', 'Spectral-only': '#ff7f0e', 'Multimodal Fusion': '#2ca02c'}

for i, r in enumerate(all_results):
    name = r['model_name']
    c = colors[name]

    # Row 1: Learning curves
    ax1 = axes[0, i]
    epochs = range(1, len(r['train_losses']) + 1)
    ax1.plot(epochs, r['train_losses'], label='Train MSE', color=c, alpha=0.7)
    ax1.plot(epochs, r['val_losses'], label='Val MSE', color=c, linestyle='--', linewidth=2)
    ax1.set_title(f'{name}', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('MSE')
    ax1.legend(frameon=True)
    ax1.grid(True, alpha=0.3)

    # Row 2: Test residual distribution
    ax2 = axes[1, i]
    # Re-evaluate to get residuals
    if name == 'Tabular-only':
        _, y_true, y_pred = evaluate(
            model_tabular.to(device), loaders_tab[2],
            nn.MSELoss(), str(device), mode='tabular',
        )
    elif name == 'Spectral-only':
        _, y_true, y_pred = evaluate(
            model_spectral.to(device), loaders_spec[2],
            nn.MSELoss(), str(device), mode='spectral',
        )
    else:
        _, y_true, y_pred = evaluate(
            model_fusion.to(device), loaders_fusion[2],
            nn.MSELoss(), str(device), mode='fusion',
        )
    residuals = y_true - y_pred
    ax2.hist(residuals, bins=40, color=c, alpha=0.7, edgecolor='white', density=True)
    mean_r, std_r = np.mean(residuals), np.std(residuals)
    ax2.axvline(0, color='black', linestyle='--', linewidth=1)
    ax2.set_title(f'Residuals  (μ={mean_r:.3f}, σ={std_r:.3f})', fontsize=11)
    ax2.set_xlabel('Error (True − Pred)')
    ax2.set_ylabel('Density')

plt.tight_layout()
plt.show()

# ── Bar chart comparison ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
names = [r['model_name'] for r in all_results]
rmse_vals = [r['test_metrics']['rmse'] for r in all_results]
r2_vals = [r['test_metrics']['r2'] for r in all_results]
bars = ax.bar(names, rmse_vals, color=[colors[n] for n in names], edgecolor='white', linewidth=1.5)
ax.set_ylabel('RMSE (↓ lower is better)', fontsize=12, fontweight='bold')
ax.set_title('Absolute Magnitude Prediction Error by Model', fontsize=13, fontweight='bold')
# Annotate bars with RMSE and R²
for bar, rmse, r2 in zip(bars, rmse_vals, r2_vals):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'RMSE={rmse:.3f}\nR²={r2:.3f}', ha='center', va='bottom', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 🔬 Scientific Interpretation & Conclusions

### Key Findings

1. **Tabular-only baseline** achieves strong performance using only 6 observables that are all distance-independent — confirming that the physical proxies (color, temperature, surface gravity, metallicity) carry substantial information about intrinsic luminosity.

2. **Spectral-only model** demonstrates that the 110-D BP/RP spectral envelope, compressed to just 5 PCA components, independently encodes enough stellar-class information to predict $M$. The spectral branch learns gravity-sensitive line broadening and metallicity-sensitive absorption features without any explicit atmospheric parameter inputs.

3. **Multimodal Fusion** combines both views via late fusion. If the non-tabular spectral information provides *complementary* (rather than redundant) signal, the fusion model should outperform both unimodal baselines.

### Why This Matters

The standard distance modulus equation $M = m - 5\log_{10}(d) + 5$ requires an accurate parallax measurement — but Gaia parallax errors grow with distance, and beyond ~5 kpc the uncertainties dominate. By learning a **direct mapping from observables → absolute magnitude**, we bypass the parallax bottleneck entirely. The spectral modality is key because it captures *what kind of star this is* (and thus its intrinsic brightness) independently of how far away it is.

### Next Steps
- Explore **early fusion** (concatenating raw features before the first hidden layer) vs. the current late fusion
- Replace PCA with the **1D Convolutional Autoencoder** (`Conv1DSpectralAutoencoder` in `utils.py`) to capture non-linear spectral features
- Extend to **DESI spectra** (higher resolution than Gaia BP/RP) for even richer gravity/metallicity diagnostics

> 📝 *End of Phase 2 deliverable. Model weights can be saved with `torch.save(results_fusion['best_state'], 'best_fusion_model.pth')` for deployment.*